In [2]:
from pyspark.sql import functions as f

df_traffic = (spark.read.format("csv").options(header=True, inferSchema=True).load('abfss://Dell_SEO_Analytics@onelake.dfs.fabric.microsoft.com/LH_SEO_Bronze.Lakehouse/Files/bronze/seo_traffic/seo_traffic.csv'))

print("Rows:", df_traffic.count())
print("Columns:", len(df_traffic.columns))

display(df_traffic.limit(10))


StatementMeta(, 533aae9e-7116-477e-bbe8-0772061fb2ed, 4, Finished, Available, Finished, False)

Rows: 50500
Columns: 14


SynapseWidget(Synapse.DataFrame, 26c9b7f1-058a-4dc7-8053-37781110b277)

In [3]:
df_traffic.printSchema()

StatementMeta(, 533aae9e-7116-477e-bbe8-0772061fb2ed, 5, Finished, Available, Finished, False)

root
 |-- traffic_id: string (nullable = true)
 |-- event_date: date (nullable = true)
 |-- country_code: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- page_id: string (nullable = true)
 |-- keyword_id: string (nullable = true)
 |-- visits: integer (nullable = true)
 |-- clicks: integer (nullable = true)
 |-- impressions: integer (nullable = true)
 |-- orders: integer (nullable = true)
 |-- revenue: double (nullable = true)
 |-- traffic_type: string (nullable = true)
 |-- updated_timestamp: timestamp (nullable = true)



In [8]:
df_traffic.groupBy("country_code").count().orderBy("country_code").show(100, False)

StatementMeta(, ac755657-50d3-4961-8b96-7525977133e6, 10, Finished, Available, Finished, False)

+-------------+-----+
|country_code |count|
+-------------+-----+
|AU           |5069 |
|BR           |5018 |
|CA           |4971 |
|DE           |5053 |
|FR           |5016 |
|IN           |5069 |
|JP           |5053 |
|MX           |5157 |
|UK           |5056 |
|US           |5033 |
|USA          |3    |
|United States|1    |
|us           |1    |
+-------------+-----+



In [4]:
df_traffic.groupBy("device_type").count().orderBy("device_type").show(100,False)

StatementMeta(, 533aae9e-7116-477e-bbe8-0772061fb2ed, 6, Finished, Available, Finished, False)

+-----------+-----+
|device_type|count|
+-----------+-----+
|Desktop    |24122|
|MOBILE     |2    |
|Mobile     |23871|
|Tablet     |2502 |
|desktop    |3    |
+-----------+-----+



In [10]:
from functools import reduce

null_counts = df_traffic.select([
    f.sum(f.when(f.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_traffic.columns
])

display(null_counts)

StatementMeta(, ac755657-50d3-4961-8b96-7525977133e6, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e09c4d0d-fbf9-4322-94af-4e237a413544)

In [5]:
duplicate_ids = (
    df_traffic
    .groupBy("traffic_id")
    .count()
    .filter(f.col("count") > 1)
)

print("Duplicate traffic IDs:", duplicate_ids.count())

display(duplicate_ids.limit(20))

StatementMeta(, 533aae9e-7116-477e-bbe8-0772061fb2ed, 7, Finished, Available, Finished, False)

Duplicate traffic IDs: 500


SynapseWidget(Synapse.DataFrame, 27bff72f-0343-4883-b684-2c8a83cfd162)

In [7]:
df_traffic.select(
    f.sum(f.when(f.col("visits") < 0, 1).otherwise(0)).alias("negative_visits"),
    f.sum(f.when(f.col("revenue") < 0, 1).otherwise(0)).alias("negative_revenue"),
    f.sum(f.when(f.col("impressions") < 0, 1).otherwise(0)).alias("negative_impressions"),
    f.sum(f.when(f.col("clicks") < 0, 1).otherwise(0)).alias("negative_clicks"),
    f.sum(f.when(f.col("orders") < 0, 1).otherwise(0)).alias("negative_orders"),
    f.sum(f.when(f.col("visits") == 0, 1).otherwise(0)).alias("zero_visits")
).show()

StatementMeta(, 533aae9e-7116-477e-bbe8-0772061fb2ed, 9, Finished, Available, Finished, False)

+---------------+----------------+--------------------+---------------+---------------+-----------+
|negative_visits|negative_revenue|negative_impressions|negative_clicks|negative_orders|zero_visits|
+---------------+----------------+--------------------+---------------+---------------+-----------+
|              3|               2|                   0|              0|              0|          3|
+---------------+----------------+--------------------+---------------+---------------+-----------+



In [8]:
df_clean = (
    df_traffic
    .withColumn(
        "country_code",
        f.when(
            f.upper(f.trim(f.col("country_code"))).isin(
                "USA", "UNITED STATES", "US"
            ),
            f.lit("US")
        )
        .otherwise(f.upper(f.trim(f.col("country_code"))))
    )
)

StatementMeta(, 533aae9e-7116-477e-bbe8-0772061fb2ed, 10, Finished, Available, Finished, False)

In [9]:
df_clean = df_clean.withColumn(
    "device_type",
    f.initcap(f.trim(f.col("device_type")))
)

display(
    df_clean
    .groupBy("device_type")
    .count()
    .orderBy("device_type")
)

StatementMeta(, 533aae9e-7116-477e-bbe8-0772061fb2ed, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c30d6042-cd1f-4779-9acd-92c2f0892702)

In [10]:
df_clean = df_clean.filter(
    (f.col("visits") >= 0) &
    (f.col("revenue") >= 0) &
    (f.col("clicks") >= 0) &
    (f.col("impressions") >= 0) &
    (f.col("orders") >= 0)
)

print("Rows after validation:", df_clean.count())

StatementMeta(, 533aae9e-7116-477e-bbe8-0772061fb2ed, 12, Finished, Available, Finished, False)

Rows after validation: 50495


In [11]:
from pyspark.sql.window import Window

window_spec = Window.partitionBy("traffic_id").orderBy(
    f.col("updated_timestamp").desc()
)

df_clean = (
    df_clean
    .withColumn("_rn", f.row_number().over(window_spec))
    .filter(f.col("_rn") == 1)
    .drop("_rn")
)

print("Rows after deduplication:", df_clean.count())

StatementMeta(, 533aae9e-7116-477e-bbe8-0772061fb2ed, 13, Finished, Available, Finished, False)

Rows after deduplication: 49995


In [12]:
print("Final Silver row count:", df_clean.count())

print("Duplicate traffic IDs:")

df_clean.groupBy("traffic_id") \
    .count() \
    .filter(f.col("count") > 1) \
    .show()

print("Invalid records:")

df_clean.filter(
    (f.col("visits") < 0) |
    (f.col("revenue") < 0) |
    (f.col("clicks") < 0) |
    (f.col("impressions") < 0) |
    (f.col("orders") < 0)
).count()

StatementMeta(, 533aae9e-7116-477e-bbe8-0772061fb2ed, 14, Finished, Available, Finished, False)

Final Silver row count: 49995
Duplicate traffic IDs:
+----------+-----+
|traffic_id|count|
+----------+-----+
+----------+-----+

Invalid records:


0

In [13]:
display(
    df_clean
    .groupBy("country_code", "device_type")
    .count()
    .orderBy("country_code", "device_type")
)

StatementMeta(, 533aae9e-7116-477e-bbe8-0772061fb2ed, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a2620a3a-f2a3-4ea0-8b73-d0bb9b144752)

In [14]:
df_clean.printSchema()

StatementMeta(, 533aae9e-7116-477e-bbe8-0772061fb2ed, 16, Finished, Available, Finished, False)

root
 |-- traffic_id: string (nullable = true)
 |-- event_date: date (nullable = true)
 |-- country_code: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- page_id: string (nullable = true)
 |-- keyword_id: string (nullable = true)
 |-- visits: integer (nullable = true)
 |-- clicks: integer (nullable = true)
 |-- impressions: integer (nullable = true)
 |-- orders: integer (nullable = true)
 |-- revenue: double (nullable = true)
 |-- traffic_type: string (nullable = true)
 |-- updated_timestamp: timestamp (nullable = true)



In [15]:
silver_table_path = "Tables/fact_seo_traffic"

(
    df_clean.write
    .format("delta")
    .mode("overwrite")
    .save(silver_table_path)
)

print("Silver table written successfully.")

StatementMeta(, 533aae9e-7116-477e-bbe8-0772061fb2ed, 17, Finished, Available, Finished, False)

Silver table written successfully.


In [16]:
df_silver = spark.read.table("fact_seo_traffic")

print("Silver table rows:", df_silver.count())
print("Silver table columns:", len(df_silver.columns))

display(df_silver.limit(10))

StatementMeta(, 533aae9e-7116-477e-bbe8-0772061fb2ed, 18, Finished, Available, Finished, False)

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `fact_seo_traffic` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.;
'UnresolvedRelation [fact_seo_traffic], [], false


In [17]:
mssparkutils.fs.rm("Tables/fact_seo_traffic", True)

print("Incorrect Delta folder removed.")

StatementMeta(, 533aae9e-7116-477e-bbe8-0772061fb2ed, 19, Finished, Available, Finished, False)

Incorrect Delta folder removed.


In [18]:
(
    df_clean.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("fact_seo_traffic")
)

print("Silver table created successfully.")

StatementMeta(, 533aae9e-7116-477e-bbe8-0772061fb2ed, 20, Finished, Available, Finished, False)

Silver table created successfully.


In [19]:
df_silver = spark.read.table("fact_seo_traffic")

print("Silver table rows:", df_silver.count())
print("Silver table columns:", len(df_silver.columns))

display(df_silver.limit(10))

StatementMeta(, 533aae9e-7116-477e-bbe8-0772061fb2ed, 21, Finished, Available, Finished, False)

Silver table rows: 49995
Silver table columns: 14


SynapseWidget(Synapse.DataFrame, e01207af-d930-4e96-adfc-1db224f18371)